# 01a_T8_acquire_standalone

Uses [LJM library in Python](https://github.com/labjack/labjack-ljm-python) only.

This is a standalone example of acquiring data from a LabJack T8 device using the
[LJM library in Python](https://github.com/labjack/labjack-ljm-python). It demonstrates how to:
1. List connected LabJack devices.
2. Open a connection to the first available device.
3. Configure and start a streaming acquisition from specified analog input channels.
4. Collect and store the acquired data in a NumPy array.

**Note 1**: Adjust the acquisition parameters (channels, sample rate, etc.) as needed
for your specific use case and device capabilities.

**Note 2**: This example uses only the LJM library, which is the recommended way to
interface with LabJack devices in Python. There is also a lower-level library called
LJME that can be used for more direct control, but LJM provides a more user-friendly
API and is generally sufficient for most applications.

**Note 3**: No use of ophyd or bluesky in this example, as it focuses on direct
interaction with the LabJack device using the LJM library. For integration with
ophyd/bluesky, see other examples in this project.


# Preamble

## Preamble for Jupyter

In [ ]:
if 0:  # if True, display full output in Jupyter, not only last result
    from IPython.core.interactiveshell import InteractiveShell

    InteractiveShell.ast_node_interactivity = "all"  # type: ignore

from IPython.display import (
    display,  # noqa: F401
    Math,  # noqa: F401
)  # to display results in Jupyter, and to print in Latex format (not only for sympy)

## Example display Latex
# display(Math('\\pi =' + f'{np.pi:.6f}'))
# display(Math('\\sin(\\theta)'))


## Preamble Main Libraries and a few usefull constants

In [ ]:
import sys  # noqa: F401
import os  # noqa: F401

from labjack import ljm
import numpy as np
import time
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go

# sys.path.append(os.environ['USERPROFILE'] + '\\workspace\\python\\libs\\wgTools')
# import wgutils as wgu
from scipy import constants

deg2rad = np.deg2rad(1)
rad2deg = np.rad2deg(1)
eps = np.finfo(np.float64).eps
hc = constants.value("inverse meter-electron volt relationship")

## Preamble for Matplotlib


In [ ]:
# # interative inline figures for JupyterLab need to be set by IPython magic commands
# # to check all the matplotlib backends, run
# %matplotlib --list
# # for vscode you want to use widget
# # For interactive figures in VS Code, use the 'widget' backend
# %matplotlib qt
%matplotlib widget
# # if you export this code to a .py file, you need to run the line below for iterative ploting in ipython
# # get_ipython().run_line_magic('matplotlib', '--list')
# # interative inline figures for JupyterLab
# # get_ipython().run_line_magic('matplotlib', 'qt5')
# # for colab you may need to install ipympl
# # !pip install ipympl
# # Then run the magic
# # %matplotlib ipympl

import matplotlib.pyplot as plt

plt.style.use("default")

params = {
    "font.size": 10,
    "legend.fontsize": "small",
    "font.family": "serif",
    "figure.facecolor": "white",
    "axes.grid": True,
    "figure.autolayout": True,
    "axes.grid.axis": "both",
    "mathtext.fontset": "stix",
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.figsize": (6, 5),
    "image.origin": "lower",
    "image.interpolation": "none",
    "image.cmap": "magma",
    "lines.marker": "o",
    "lines.linestyle": "-",
}

plt.rcParams.update(params)

## Preamble Plotly

In [ ]:
# import plotly.colors

# colors_plotly = plotly.colors.DEFAULT_PLOTLY_COLORS


# def custom_layout(xlabel=None, ylabel=None, title=None, template="seaborn"):
#     return go.Layout(
#         template=template,  # Set the template
#         width=800,  # Set the figure width
#         height=600,  # Set the figure height
#         font=dict(family="Georgia", size=14, color="#3B3B3B"),  # General font settings
#         title=dict(
#             text=title,
#             font=dict(size=24),
#             x=0.5,
#             xanchor="center",
#         ),
#         xaxis_title=dict(text=xlabel, font=dict(size=18)),
#         yaxis_title=dict(text=ylabel, font=dict(size=18)),
#         xaxis=dict(tickfont=dict(size=16)),  # Set X-axis tick font size
#         yaxis=dict(tickfont=dict(size=16)),  # Set Y-axis tick font size
#     )


# # initialize figures with something like
# # plotly_fig = go.Figure(layout=custom_layout("Frequency [Hz]", "Amplitude [Volts]", fname))


## Local Functions

In [ ]:
def datenow_str():
    return datetime.now().strftime("%Y%m%d_%H%M%S")


def check_out_of_range(flat_data: list, channel: dict, verbose: bool = True):
    """
    Checks for out-of-range values in the acquired data and prints warnings if any are found.

    Parameters
    ----------
    flat_data : list
        A flat list of acquired data values, as provided by ljm.eStreamRead(handle), that are ordered by channel (e.g., [ch1_scan1, ch2_scan1, ch3_scan1, ch1_scan2, ch2_scan2, ...]).
    actual_range : list
        A list of actual voltage ranges for each channel (e.g., [±11, ±9.6, ±4.8, ...]).

    See also
    --------
    Ranges and actual ranges are listed at https://support.labjack.com/docs/a-3-3-3-t8-signal-range-t-series-datasheet

    """

    actual_range = channel["actual_range"]
    channels = channel["name"]
    _ncurves = len(actual_range)

    _out_of_range = []

    for _i in range(_ncurves):
        _ch_data = flat_data[_i::_ncurves]  # extract data for this channel
        _max_abs = max(abs(np.array(_ch_data)))
        if _max_abs > actual_range[_i]:
            if verbose:
                print(
                    f"[WARNING] OUT-OF-RANGE: {channels[_i]} max absolute value {_max_abs:.3f} V is higher than actual channel range of ±{actual_range[_i]:.3f} V."
                )
            _out_of_range.append(True)

        else:
            if verbose:
                print(f"[INFO] {channels[_i]} is within the expected range of ±{actual_range[_i]:.3f} V.")
            _out_of_range.append(False)

    return _out_of_range

# Setup Script Flags

In [ ]:
save_plot = False
close_connections_at_end = False

# Setup Hardware

## List Labjack Devices

In [ ]:
print("\n[INFO] ### Searching for connected Devices...")
try:
    res = ljm.listAllS("ANY", "ANY")
    print(f"[INFO] Devices found: {res[0]}")
    for i in range(res[0]):
        print(f"\t- Device {i}: {res[1][i]}, Connection: {res[2][i]}, Serial: {res[3][i]}, IP: {res[4][i]},")
except ljm.LJMError as e:
    print(f"[ERROR] Failed to list devices: {e}")
    res = None

In [ ]:
if False:  # set to True to stop here and inspect the device list before proceeding
    print("\n[INFO] Raw results from ljm.listAllS('ANY', 'ANY'):")
    print(res)

## Opening connection to device with Serial number


In [ ]:
_serial_to_connection = res[3][0]  # open by serial number
print(f"\n[INFO] Opening connection to device with Serial: {_serial_to_connection}.")
handle = ljm.openS("ANY", "ANY", _serial_to_connection)  # open by serial number
print("[INFO] Opened connection.")

info = ljm.getHandleInfo(handle)
print("[INFO] Device info from handle:")
print(f"[INFO] Device type: {info[0]}, Connection: {info[1]}, Serial: {info[2]}, IP: {info[3]}, Port: {info[4]}")

## Acquisition Set Points

In [ ]:
# NUM_SAMPLES = 40_000  # total points per channel
SAMPLE_RATE = 1000  # Hz, The T8 has a maximum scan rate of 40 ksamples/second. Unlike other T-series devices, the scan rate does not depend on how many addresses are sampled per scan since all analog inputs are sampled simultaneously.
# CHANNEL_NAMES = ["AIN0", "AIN1", "AIN2", "AIN4", "AIN5"]
CHANNEL_NAMES = ["AIN0", "AIN1"]
# RANGES_PER_CHANNEL = [10.0] * len(channels)  # set all channels to ±10V range
# RANGES_PER_CHANNEL = [10.0, 10.0, 3.0, 1.0, 0.010]  # set each channel to a specific range
RANGES_PER_CHANNEL = [10.0, 10.0]  # set each channel to a specific range
# channels = ["AIN0"]  # use names to avoid ambiguity
SCANS_PER_READ = 1000  # chunk size per eStreamRead (reasonable default)


In [ ]:
channels = {}
channels["name"] = CHANNEL_NAMES
channels["range"] = RANGES_PER_CHANNEL

num_ch = len(channels["name"])
channels

## Set the range for each channel.

This is important to ensure you get the best resolution for your expected signal levels.

AIN range options for T8 (from LabJack documentation):

Ranges are: 

±11 V, ±9.6 V, ±4.8 V, ±2.4 V, ±1.2 V,

±0.6 V, ±0.3 V, ±0.15 V,

±0.075 V, ±0.036 V, ±0.018 V

In [ ]:
actual_range = []
for i, ch in enumerate(channels["name"]):
    print(f"[INFO] Setting range for channel {ch} to ±{RANGES_PER_CHANNEL[i]} V")
    ljm.eWriteName(handle, f"{ch}_RANGE", RANGES_PER_CHANNEL[i])

    actual_range.append(ljm.eReadName(handle, f"{ch}_RANGE"))
    print(f"[INFO] Actual range set for {ch}: ±{actual_range[-1]:.6f} V")

channels["actual_range"] = actual_range
print(f"\n[INFO] SSetting up stream at {SAMPLE_RATE} Hz from {num_ch} channels...")

In [ ]:
# %% map names to addresses. Needed for streaming API. Also get types for reference.
channels["aAddresses"], channels["aTypes"] = ljm.namesToAddresses(num_ch, channels["name"])
# num_ch = len(channels["aAddresses"])

print("\n[INFO] Channels info:")

print("\t- aAddresses:", channels["aAddresses"])
print("\t- aTypes:", channels["aTypes"])
# reference for aTypes:https://support.labjack.com/docs/ewriteaddresses-ljm-user-s-guide#aTypes-[in]
# Values according to LJM library constants:
# 0 = UINT16, 1 = UINT32, 2 = INT32, 3 = FLOAT32, 98 = STRING, 99 = BYTE

In [ ]:
channels

# Run stream

Note that this start and stop the stream. If we dont stop, data will continuouly be saved to the buffer.

## Settings

In [ ]:
NUM_SAMPLES = 10_000  # total points per channel
acq_time = NUM_SAMPLES / SAMPLE_RATE

# Settings for real-time plotting
PLOT_UPDATE_EVERY = 1  # Update plot every N reads (to avoid too frequent updates)
PLOT_WINDOW_SECONDS = 10.0  # Show last N seconds of data

## Main Stream Loop

In [ ]:
def _start_real_time_acquisition(
    handle, SCANS_PER_READ, num_ch, channels, SAMPLE_RATE, PLOT_WINDOW_SECONDS, PLOT_UPDATE_EVERY, NUM_SAMPLES, acq_time
):

    print(f"\n[INFO] Starting REAL-TIME acquisition: {NUM_SAMPLES} scans at {SAMPLE_RATE} Hz from {num_ch} channels...")
    print("[INFO] 🛑 Press Ctrl+C to stop acquisition gracefully\n")

    start_t = time.time()
    actual_rate = ljm.eStreamStart(handle, SCANS_PER_READ, num_ch, channels["aAddresses"], SAMPLE_RATE)
    print(f"[INFO] eStreamStart STARTED. Requested rate: {SAMPLE_RATE} Hz, Actual rate: {actual_rate:.2f} Hz")

    all_flat = []
    scan_interval = 1.0 / actual_rate  # time between scans in seconds

    # Real-time plotting variables
    read_count = 0
    plot_data = {ch: [] for ch in channels["name"]}  # Store recent data for each channel
    plot_time = {ch: [] for ch in channels["name"]}  # Store corresponding time stamps
    acquisition_start_time = time.time()  # Reference time for plotting

    print(f"[INFO] Plot will show last {PLOT_WINDOW_SECONDS:.1f} seconds of data")

    _counter = 1
    interrupted = False

    try:
        while True:
            # Check completion conditions
            scans_collected = len(all_flat) // num_ch
            if scans_collected >= NUM_SAMPLES:
                break

            elapsed = time.time() - start_t
            if elapsed >= acq_time * 2:
                print(f"[WARNING] Timeout reached ({elapsed:.2f}s)")
                break

            # Progress indicator
            if scans_collected >= (NUM_SAMPLES // 10) * _counter:
                _counter += 1
                print(f"\nProgress: {scans_collected / NUM_SAMPLES * 100:.1f}% ", end="", flush=True)

            # Read data from LabJack
            ret = ljm.eStreamRead(handle)
            print(".", end="", flush=True)

            if ret and ret[0]:
                all_flat.extend(ret[0])

                # ===== REAL-TIME PLOTTING =====
                read_count += 1
                if read_count % PLOT_UPDATE_EVERY == 0:  # Update plot every N reads
                    # Process new data for plotting
                    new_scans = len(ret[0]) // num_ch
                    current_time = time.time()

                    MAX_POINTS_PLOT = 100
                    if new_scans > MAX_POINTS_PLOT:
                        _stride = new_scans // MAX_POINTS_PLOT
                    else:
                        _stride = 1

                    for scan_idx in range(0, new_scans, _stride):
                        # Calculate time for this scan (approximate)
                        scan_time = current_time - acquisition_start_time + (scan_idx * scan_interval)

                        for ch_idx, ch_name in enumerate(channels["name"]):
                            sample_idx = scan_idx * num_ch + ch_idx
                            value = ret[0][sample_idx]

                            plot_data[ch_name].append(value)
                            plot_time[ch_name].append(scan_time)

                    # Keep only data from the last PLOT_WINDOW_SECONDS
                    current_plot_time = current_time - acquisition_start_time
                    cutoff_time = current_plot_time - PLOT_WINDOW_SECONDS

                    for ch_name in channels["name"]:
                        # Remove old data points
                        while plot_time[ch_name] and plot_time[ch_name][0] < cutoff_time:
                            plot_time[ch_name].pop(0)
                            plot_data[ch_name].pop(0)

                    # Update the plot
                    try:
                        for i, ch_name in enumerate(channels["name"]):
                            if len(plot_data[ch_name]) > 1:
                                x_data = plot_time[ch_name]
                                y_data = plot_data[ch_name]

                                lines[i].set_data(x_data, y_data)

                                # Set x-axis to show the time window
                                if x_data:
                                    axes[i].set_xlim(
                                        max(0, current_plot_time - PLOT_WINDOW_SECONDS), current_plot_time + 0.5
                                    )  # Small buffer on the right

                                # Auto-scale y-axis if needed
                                if y_data:
                                    y_min, y_max = min(y_data), max(y_data)
                                    margin = (y_max - y_min) * 0.1 if y_max != y_min else 0.1
                                    axes[i].set_ylim(y_min - margin, y_max + margin)

                        fig.canvas.draw()
                        fig.canvas.flush_events()

                    except Exception as e:
                        print(f"[Plot Error]: {e}")
                # ===== END REAL-TIME PLOTTING =====

            time.sleep(0.0001)

    except KeyboardInterrupt:
        print("\n\n[INFO] 🛑 Keyboard interrupt received (Ctrl+C)")
        print("[INFO] Stopping acquisition gracefully...")
        interrupted = True

    finally:
        # Always stop the stream and clean up
        try:
            ljm.eStreamStop(handle)
            print("[INFO] ✅ eStreamStart STOPPED.")
        except Exception as e:
            print(f"[WARNING] Error stopping stream: {e}")

    elapsed = time.time() - start_t
    scans_collected = len(all_flat) // num_ch

    if interrupted:
        print(f"\n[INFO] 🛑 Acquisition interrupted by user after {elapsed:.3f}s")
    else:
        print(f"\n\n[INFO] ✅ Acquisition completed in {elapsed:.3f}s")

    print(f"[INFO] Total scans requested: {NUM_SAMPLES}, collected: {scans_collected}")
    print(f"[INFO] Requested rate: {SAMPLE_RATE} Hz, Actual rate: {actual_rate:.2f} Hz")
    print(f"[INFO] Elapsed time: {elapsed:.3f} s. Acquisition time: {scans_collected * scan_interval:.3f} s")

    if scans_collected > 0:
        print("[INFO] 📊 Data collected successfully! Use cells below for analysis.")

    return all_flat

### Set up matplotlib graphs

In [ ]:
def setup_realtime_plot(channels, plot_window_seconds):
    """
    Setup real-time plot for LabJack data acquisition.

    Parameters:
    -----------
    channels : dict
        Dictionary containing channel information with 'name' and 'actual_range' keys
    plot_window_seconds : float
        Time window in seconds to display on the plot

    Returns:
    --------
    tuple
        (fig, axes, lines) - matplotlib figure, axes array, and lines array
    """
    # Setup real-time plot
    print("Setting up real-time plot...")

    # Create figure and axes
    fig, axes = plt.subplots(len(channels["name"]), 1, figsize=(6, 3 * len(channels["name"])))
    if len(channels["name"]) == 1:
        axes = [axes]  # Make it a list for consistency

    # Initialize empty lines for each channel
    lines = []
    for i, (ax, ch_name) in enumerate(zip(axes, channels["name"])):
        (line,) = ax.plot([], [], ".-", linewidth=1, markersize=2)
        lines.append(line)
        ax.set_title(f"{ch_name} - Real-time (Last {plot_window_seconds}s)")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Voltage (V)")
        ax.grid(True, alpha=0.3)

        # Set y-limits based on channel range
        range_val = channels["actual_range"][i]
        ax.set_ylim(-range_val * 1.1, range_val * 1.1)

        # Set initial x-axis
        ax.set_xlim(0, plot_window_seconds)

        # Prevent matplotlib from using offsets on axes
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.1f}"))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.3f}"))

    plt.tight_layout()
    plt.ion()  # Turn on interactive mode
    plt.show()

    print(f"Real-time plot ready! Showing last {plot_window_seconds} seconds of data.")

    return fig, axes, lines


# Call the function with existing variables
fig, axes, lines = setup_realtime_plot(channels, PLOT_WINDOW_SECONDS)

### Run the main loop

In [ ]:
all_flat = _start_real_time_acquisition(
    handle, SCANS_PER_READ, num_ch, channels, SAMPLE_RATE, PLOT_WINDOW_SECONDS, PLOT_UPDATE_EVERY, NUM_SAMPLES, acq_time
)

#### Inspect Results

In [ ]:
all_flat

## Check Out-of-range

In [ ]:
print("\n[INFO] Checking for out-of-range values...")

if any(check_out_of_range(all_flat, channels)):
    print("[WARNING] Some values are out of range!")

## Convert to numpy, trim partial tail, reshape

In [ ]:
scans_collected = len(all_flat) // num_ch
_arr_data = np.asarray(all_flat, dtype=float).reshape(scans_collected, num_ch)
# If you asked for a specific total, trim/keep only the requested scans
if _arr_data.shape[0] > NUM_SAMPLES:
    _arr_data = _arr_data[:NUM_SAMPLES, :]

# Generate timestamps for each scan
times = np.arange(_arr_data.shape[0]) * 1 / SAMPLE_RATE

# Combine times with data
_arr_data = np.hstack([times.reshape(-1, 1), _arr_data])

print(f"[INFO] Collected {_arr_data.shape[0]} scans x {_arr_data.shape[1] - 1} channels (+ time column)")
print(f"[INFO] Time range: {times[0]:.6f} to {times[-1]:.6f} seconds (session time)")


## Convert to pandas

In [ ]:
df = pd.DataFrame(_arr_data, columns=["Time"] + channels["name"])
_arr_data = None
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# Post Processing

In [ ]:
print("\n[POSTPROCESSING] Post Processing STARTED")

MAX_POINTS_PLOT = 100
if df.shape[0] > MAX_POINTS_PLOT:
    _stride = df.shape[0] // MAX_POINTS_PLOT
else:
    _stride = 1

_chs2plot = df.columns[1:]
# _chs2plot = [
#     "AIN0",  #
#     "AIN1",
#     "AIN2",
#     "AIN4",
#     "AIN5",
# ]

print("\n[POSTPROCESSING] Plotting data with Matplotlib...")
fig, ax = plt.subplots(figsize=(6, 4))
times = df["Time"]
for _column in _chs2plot:
    if _column in df.columns:
        plt.plot(times[::_stride], df[_column][::_stride], ".-", label=f"{_column}")

plt.title("LabJack Data Acquisition - Complete Dataset")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.legend()
plt.grid(True, alpha=0.3)

# Prevent matplotlib from using offsets on axes
ax = plt.gca()
ax.ticklabel_format(useOffset=False, style="plain")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.4f}"))

# Save figure and open in VS Code webview
if save_plot:
    plt.savefig(f"Results\\{datenow_str()}_labjack_acquisition_plot.png", dpi=300, bbox_inches="tight")
    print(f"[POSTPROCESSING] Plot saved to .\\Results\\{datenow_str()}_labjack_acquisition_plot.png\n")

plt.show()

In [ ]:
# print("\n[POSTPROCESSING] Post Processing STARTED")


# print("\n[POSTPROCESSING] Plotting data with Plotly...")
# fig = go.Figure()
# times = df["Time"]
# for i in range(1, df.shape[1]):
#     fig.add_trace(go.Scatter(x=times, y=df.iloc[:, i], mode="lines", name=f"{df.columns[i]}"))

# fig.update_layout(
#     title="LabJack Data Acquisition",
#     xaxis_title="Time (s)",
#     yaxis_title="Voltage (V)",
#     hovermode="x unified",
#     template="plotly",
# )
# # Save HTML and open in VS Code webview
# html_file = f"Results\\{datenow_str()}_labjack_acquisition_plot.html"

# if save_plot:
#     fig.write_html(html_file)
#     print(f"[POSTPROCESSING] Plot saved to .\\{html_file}\n")

# fig.show()

# END - Close connections

In [ ]:
if close_connections_at_end:
    # if True:
    try:
        ljm.close(handle)
        print("[INFO] Closed connection to hardware.")
    except Exception as e:
        print("[ERROR] Close error:", e)

    print("\n[INFO] ### Acquisition complete ###\n")
